In [5]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY=os.getenv("OPEN_ROUTER_API_KEY")

if OPENAI_API_KEY:
    print("OPEN_ROUTER_API_KEY is set.")
else:
    print("OPEN_ROUTER_API_KEY is not set.")

OPEN_ROUTER_API_KEY is set.


**1st stepLoad pdf**

In [3]:
text_data = PyPDFLoader("NovaS.pdf").load()

# Changing Metadata of the document
for page in text_data:
    page.metadata["source"] = "NovaS.pdf"

text_data

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by d

**2nd step creating chunks**

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

chunks = splitter.split_documents(text_data)
len(chunks)

101

In [23]:
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='and technology company that has grown gradually over the years. The organization was'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '

**3rd creating embdding and storing in chroma db**

In [7]:


openrouter_base_url = "https://openrouter.ai/api/v1"
embed_model = OpenAIEmbeddings(model="openai/text-embedding-3-small",api_key=OPENAI_API_KEY,base_url=openrouter_base_url,
)
#using paid model, free didnt work.


In [ ]:

chroma_db = Chroma.from_documents(chunks, embed_model, persist_directory="./chroma_db")



**4th connection and retreival**

In [8]:
chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)

C:\Users\HP\AppData\Local\Temp\ipykernel_10540\3816121341.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


In [9]:
chroma_db_con.similarity_search("What is the main topic of the document?", k=2)

[Document(metadata={'total_pages': 3, 'page': 1, 'producer': 'Microsoft® Word for Microsoft 365', 'moddate': '2026-03-31T11:24:15-03:00', 'creator': 'Microsoft® Word for Microsoft 365', 'author': 'Ansh Lamba', 'page_label': '2', 'source': 'NovaS.pdf', 'creationdate': '2026-03-31T11:24:15-03:00'}, page_content='new clients, the company spent time improving documentation, building reusable code'),
 Document(metadata={'source': 'NovaS.pdf', 'creator': 'Microsoft® Word for Microsoft 365', 'author': 'Ansh Lamba', 'creationdate': '2026-03-31T11:24:15-03:00', 'total_pages': 3, 'page_label': '2', 'producer': 'Microsoft® Word for Microsoft 365', 'moddate': '2026-03-31T11:24:15-03:00', 'page': 1}, page_content='Technologies used that time to improve its internal systems. Instead of focusing only on')]

rag part is done, now we can use it to enrich query and make llm call.

In [10]:
'''llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
user_query = input("Enter your question: ")

rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
rel_chunks_content = str(rel_chunks_content)

llm.invoke(f"{user_query}, Use the following context to answer the question: {rel_chunks_content}")'''

'llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)\nuser_query = input("Enter your question: ")\n\nrel_chunks = chroma_db_con.similarity_search(user_query, k=3)\n\nrel_chunks_content = []\nfor i, chunk in enumerate(rel_chunks):\n    rel_chunks_content.append(chunk.page_content)\nrel_chunks_content = str(rel_chunks_content)\n\nllm.invoke(f"{user_query}, Use the following context to answer the question: {rel_chunks_content}")'